قدم ۱: ساخت فولدرهای pipeline

In [1]:
from pathlib import Path
import os

project_root = Path("..").resolve()

folders = [
    project_root / "src" / "embeddings",
    project_root / "src" / "retrieval",
    project_root / "src" / "generation",
    project_root / "src" / "cache",
    project_root / "src" / "pipeline",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    init_file = folder / "__init__.py"
    init_file.touch(exist_ok=True)

print("Project root:", project_root)
print("Folders created:")
for folder in folders:
    print("-", folder)

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
Folders created:
- /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/embeddings
- /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/retrieval
- /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/generation
- /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/cache
- /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/pipeline


قدم ۲: ساخت arvan_embedder.py

In [2]:
embedder_code = r'''
import os
from openai import OpenAI


class ArvanEmbedder:
    def __init__(
        self,
        api_key: str | None = None,
        base_url: str | None = None,
        model: str = "text-embedding-3-large"
    ):
        self.api_key = api_key or os.getenv("ARVAN_AI_API_KEY")
        self.base_url = base_url or os.getenv("ARVAN_EMBEDDING_URL")
        self.model = model

        if not self.api_key:
            raise ValueError("ARVAN_AI_API_KEY is not set")

        if not self.base_url:
            raise ValueError("ARVAN_EMBEDDING_URL is not set")

        self.client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url
        )

    def embed_text(self, text: str) -> list[float]:
        response = self.client.embeddings.create(
            model=self.model,
            input=text
        )

        return response.data[0].embedding

    def embed_query(self, query: str) -> list[float]:
        return self.embed_text(query)
'''

embedder_file = project_root / "src" / "embeddings" / "arvan_embedder.py"

with open(embedder_file, "w", encoding="utf-8") as f:
    f.write(embedder_code)

print("Created:", embedder_file)

Created: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/embeddings/arvan_embedder.py


قدم ۳: تست ماژول Embedding

In [3]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env")

from src.embeddings.arvan_embedder import ArvanEmbedder

embedder = ArvanEmbedder()

test_vector = embedder.embed_query("سواد مالی یعنی چه؟")

print("Embedding length:", len(test_vector))
print("First 5 values:", test_vector[:5])

Embedding length: 3072
First 5 values: [-0.036712646484375, -0.006290435791015625, -0.001895904541015625, -0.039886474609375, 0.01422119140625]


قدم ۴: ساخت qdrant_retriever.py

In [4]:
retriever_code = r'''
from pathlib import Path
from qdrant_client import QdrantClient


class QdrantRetriever:
    def __init__(
        self,
        vectorstore_path: str | Path,
        collection_name: str
    ):
        self.vectorstore_path = Path(vectorstore_path)
        self.collection_name = collection_name

        if not self.vectorstore_path.exists():
            raise FileNotFoundError(f"Vectorstore path not found: {self.vectorstore_path}")

        self.client = QdrantClient(
            path=str(self.vectorstore_path)
        )

    def search(self, query_vector: list[float], top_k: int = 5) -> list[dict]:
        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=top_k,
            with_payload=True
        )

        retrieved_chunks = []

        for point in results.points:
            payload = point.payload or {}

            retrieved_chunks.append({
                "score": point.score,
                "chunk_id": payload.get("chunk_id"),
                "article_id": payload.get("article_id"),
                "chunk_index": payload.get("chunk_index"),
                "title": payload.get("title"),
                "url": payload.get("url"),
                "text": payload.get("text"),
                "char_count": payload.get("char_count")
            })

        return retrieved_chunks
'''
 
retriever_file = project_root / "src" / "retrieval" / "qdrant_retriever.py"

with open(retriever_file, "w", encoding="utf-8") as f:
    f.write(retriever_code)

print("Created:", retriever_file)

Created: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/retrieval/qdrant_retriever.py


In [5]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env")

print("Project root:", project_root)
print("Config exists:", (project_root / "config" / "settings.py").exists())

Project root: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance
Config exists: True


In [6]:
try:
    retriever.client.close()
    print("Previous retriever client closed")
except NameError:
    print("No previous retriever found")
except Exception as e:
    print("Could not close previous retriever:", e)

No previous retriever found


قدم ۵: تست ماژول Retrieval

In [7]:
import importlib
import config.settings as settings

importlib.reload(settings)

from src.retrieval.qdrant_retriever import QdrantRetriever

try:
    retriever.client.close()
    print("Previous retriever client closed")
except NameError:
    print("No previous retriever found")
except Exception as e:
    print("Could not close previous retriever:", e)

QDRANT_LOCAL_PATH = Path(settings.DATA_VECTORSTORE_PATH) / "qdrant_local"

retriever = QdrantRetriever(
    vectorstore_path=QDRANT_LOCAL_PATH,
    collection_name=settings.COLLECTION_NAME
)

results = retriever.search(
    query_vector=test_vector,
    top_k=5
)

print("Results:", len(results))

for i, item in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Score:", item["score"])
    print("Chunk ID:", item["chunk_id"])
    print("Title:", item["title"])
    print("Text preview:")
    print(item["text"][:500])

No previous retriever found
Results: 5
Rank: 1
Score: 0.5363671821603111
Chunk ID: 5C92BB0D_3
Title: فیل سواد مالی
Text preview:
یا سرمایه‌گذاری را تمام آن چیزی معرفی کنیم که او در آینده لازم دارد. روی سخن با کسانی نیست که در برندسازی از نام «سواد مالی» بهره خود را می‌برند؛ مخاطب آن‌هایی هستند که دغدغه تربیت نسل آینده را دارند. در کف هر یک اگر شمعی بدی / اختلاف از گفت‌شان بیرون شدی دانش، نور است و اگر پای تخصص در سواد مالی به میان آید، بسیاری از آسیب‌های برآمده از نگاه بخشی و انحرافی به سواد مالی رخت برمی‌بندد. این تعهدی است که در برابر آیندگان داریم.

داستان فیلِ در تاریکی، از مشهورترین داستان‌های مثنوی مولوی است؛ فیلی ک
Rank: 2
Score: 0.52629458209961
Chunk ID: BD65A55B_1
Title: سیر مطالعاتی سواد مالی؛ گام نخست
Text preview:
دانش، دارایی اصلی بشر برای رسیدن به اهدافش است. توصیه به دانش‌اندوزی در همه جوامع و فرهنگ‌ها دیده می‌شود. چراکه هیچ انسانی بدون دانش نمی‌تواند چشم‌اندازی روشن از خود،جامعه و نسل آینده و مسیر پیش رویش داشته باشد. «سواد مالی» یکی از دانش‌های مفید و ضروری برای همۀ م

قدم ۶: ساخت arvan_generator.py

In [8]:
generator_code = r'''
import os
from openai import OpenAI


class ArvanGenerator:
    def __init__(
        self,
        api_key: str | None = None,
        base_url: str | None = None,
        model: str | None = None,
        temperature: float = 0.2
    ):
        self.api_key = api_key or os.getenv("ARVAN_CHAT_API_KEY") or os.getenv("ARVAN_AI_API_KEY")
        self.base_url = base_url or os.getenv("ARVAN_CHAT_URL")
        self.model = model or os.getenv("ARVAN_CHAT_MODEL") or "Qwen3-30B-A3B"
        self.temperature = temperature

        if not self.api_key:
            raise ValueError("ARVAN_CHAT_API_KEY or ARVAN_AI_API_KEY is not set")

        if not self.base_url:
            raise ValueError("ARVAN_CHAT_URL is not set")

        self.client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url
        )

    def generate_answer(self, question: str, context: str) -> str:
        system_prompt = """
تو یک دستیار مالی فارسی هستی.
فقط بر اساس منابع داده‌شده پاسخ بده.
اگر پاسخ در منابع نبود، دقیقاً بگو: «در منابع موجود پاسخ کافی پیدا نشد.»
پاسخ را کوتاه، دقیق و روان بنویس.
"""

        user_prompt = f"""
سؤال:
{question}

منابع:
{context}

پاسخ:
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()}
            ],
            temperature=self.temperature
        )

        return response.choices[0].message.content.strip()
'''

generator_file = project_root / "src" / "generation" / "arvan_generator.py"

with open(generator_file, "w", encoding="utf-8") as f:
    f.write(generator_code)

print("Created:", generator_file)

Created: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/generation/arvan_generator.py


قدم ۷: ساخت context و تست generator

In [9]:
from src.generation.arvan_generator import ArvanGenerator

def build_context(retrieved_chunks):
    context_parts = []

    for i, item in enumerate(retrieved_chunks, start=1):
        part = f"""
[منبع {i}]
عنوان: {item["title"]}
آدرس: {item["url"]}
شناسه چانک: {item["chunk_id"]}

متن:
{item["text"]}
"""
        context_parts.append(part.strip())

    separator = "\n\n" + ("-" * 80) + "\n\n"
    return separator.join(context_parts)


question = "سواد مالی یعنی چه؟"
context = build_context(results)

generator = ArvanGenerator()

answer = generator.generate_answer(
    question=question,
    context=context
)

print("Question:", question)
print()
print("Answer:")
print(answer)

Question: سواد مالی یعنی چه؟

Answer:
سواد مالی، کسب دانش کافی در مورد مدیریت امور مالی، داشتن نگرش صحیح و اجرای درست آموزه‌ها برای رسیدن به امنیت و آزادی مالی است. هر کسی باید بتواند از پس امور مالی خود برآید و تصمیم‌های درستی در مورد پس‌انداز، سرمایه‌گذاری و مدیریت بدهی‌ها بگیرد.


قدم ۸: ساخت rag_cache.py

In [10]:
cache_code = r'''
import json
import hashlib
from pathlib import Path
from datetime import datetime


class RAGCache:
    def __init__(self, cache_file: str | Path):
        self.cache_file = Path(cache_file)
        self.cache_file.parent.mkdir(parents=True, exist_ok=True)

        if not self.cache_file.exists():
            self.save_cache({})

    def normalize_question(self, question: str) -> str:
        return " ".join(question.strip().lower().split())

    def make_question_hash(self, question: str) -> str:
        normalized = self.normalize_question(question)
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    def load_cache(self) -> dict:
        with open(self.cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    def save_cache(self, cache: dict):
        with open(self.cache_file, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)

    def get(self, question: str):
        question_hash = self.make_question_hash(question)
        cache = self.load_cache()
        return cache.get(question_hash)

    def set(
        self,
        question: str,
        answer: str,
        sources: list,
        retrieved_chunks: list,
        top_k: int,
        embedding_model: str,
        generation_model: str
    ):
        question_hash = self.make_question_hash(question)
        cache = self.load_cache()

        cache[question_hash] = {
            "question": question,
            "question_hash": question_hash,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
            "top_k": top_k,
            "embedding_model": embedding_model,
            "generation_model": generation_model,
            "created_at": datetime.now().isoformat()
        }

        self.save_cache(cache)

        return cache[question_hash]
'''

cache_file = project_root / "src" / "cache" / "rag_cache.py"

with open(cache_file, "w", encoding="utf-8") as f:
    f.write(cache_code)

print("Created:", cache_file)

Created: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/cache/rag_cache.py


قدم ۹: تست RAGCache

In [11]:
from src.cache.rag_cache import RAGCache

cache_path = project_root / "data" / "cache" / "rag_cache.json"

rag_cache = RAGCache(cache_file=cache_path)

cached_item = rag_cache.get("سواد مالی یعنی چه؟")

print("Cache file:", cache_path)
print("Cached item found:", cached_item is not None)

if cached_item:
    print("Question:", cached_item["question"])
    print("Answer preview:", cached_item["answer"][:300])

Cache file: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/data/cache/rag_cache.json
Cached item found: True
Question: سواد مالی یعنی چه؟
Answer preview: سواد مالی، کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی است.  
هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است.  
سواد مالی شامل مفاهیمی مانند خرج و پس‌انداز، اعتبار و بدهی، سرمایه‌گذاری و مدیریت ریسک می‌شود.


قدم ۱۰: ساخت rag_pipeline.py

In [12]:
pipeline_code = r'''
from pathlib import Path

from src.embeddings.arvan_embedder import ArvanEmbedder
from src.retrieval.qdrant_retriever import QdrantRetriever
from src.generation.arvan_generator import ArvanGenerator
from src.cache.rag_cache import RAGCache


class RAGPipeline:
    def __init__(
        self,
        vectorstore_path: str | Path,
        collection_name: str,
        cache_file: str | Path | None = None,
        embedding_model: str = "text-embedding-3-large",
        generation_model: str = "Qwen3-30B-A3B",
        use_cache: bool = True
    ):
        self.embedding_model = embedding_model
        self.generation_model = generation_model
        self.use_cache = use_cache

        self.embedder = ArvanEmbedder(
            model=self.embedding_model
        )

        self.retriever = QdrantRetriever(
            vectorstore_path=vectorstore_path,
            collection_name=collection_name
        )

        self.generator = ArvanGenerator(
            model=self.generation_model
        )

        self.cache = RAGCache(cache_file) if cache_file else None

    def build_context(self, retrieved_chunks: list[dict]) -> str:
        context_parts = []

        for i, item in enumerate(retrieved_chunks, start=1):
            part = f"""
[منبع {i}]
عنوان: {item["title"]}
آدرس: {item["url"]}
شناسه چانک: {item["chunk_id"]}

متن:
{item["text"]}
"""
            context_parts.append(part.strip())

        separator = "\n\n" + ("-" * 80) + "\n\n"
        return separator.join(context_parts)

    def format_sources(self, retrieved_chunks: list[dict]) -> list[dict]:
        sources = []
        seen = set()

        for item in retrieved_chunks:
            key = item.get("url") or item.get("title")

            if key in seen:
                continue

            seen.add(key)

            sources.append({
                "title": item.get("title"),
                "url": item.get("url"),
                "chunk_id": item.get("chunk_id"),
                "score": item.get("score")
            })

        return sources

    def ask(self, question: str, top_k: int = 5) -> dict:
        if self.use_cache and self.cache:
            cached_item = self.cache.get(question)

            if cached_item:
                cached_item["from_cache"] = True
                return cached_item

        query_vector = self.embedder.embed_query(question)

        retrieved_chunks = self.retriever.search(
            query_vector=query_vector,
            top_k=top_k
        )

        context = self.build_context(retrieved_chunks)

        answer = self.generator.generate_answer(
            question=question,
            context=context
        )

        sources = self.format_sources(retrieved_chunks)

        result = {
            "question": question,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
            "top_k": top_k,
            "embedding_model": self.embedding_model,
            "generation_model": self.generation_model,
            "from_cache": False
        }

        if self.use_cache and self.cache:
            result = self.cache.set(
                question=question,
                answer=answer,
                sources=sources,
                retrieved_chunks=retrieved_chunks,
                top_k=top_k,
                embedding_model=self.embedding_model,
                generation_model=self.generation_model
            )
            result["from_cache"] = False

        return result

    def close(self):
        try:
            self.retriever.client.close()
        except Exception:
            pass
'''

pipeline_file = project_root / "src" / "pipeline" / "rag_pipeline.py"

with open(pipeline_file, "w", encoding="utf-8") as f:
    f.write(pipeline_code)

print("Created:", pipeline_file)

Created: /Users/macbookpro/AMIR_DATA/00_Project/VSCode__project/RAG_finance/src/pipeline/rag_pipeline.py


قدم ۱۱: تست کامل Pipeline

In [14]:
from src.pipeline.rag_pipeline import RAGPipeline
import config.settings as settings

QDRANT_LOCAL_PATH = Path(settings.DATA_VECTORSTORE_PATH) / "qdrant_local"
CACHE_FILE = project_root / "data" / "cache" / "rag_cache.json"

try:
    rag.close()
    print("Previous RAG pipeline closed")
except NameError:
    print("No previous RAG pipeline found")
except Exception as e:
    print("Could not close previous RAG pipeline:", e)

try:
    retriever.client.close()
    print("Previous standalone retriever closed")
except NameError:
    print("No previous standalone retriever found")
except Exception as e:
    print("Could not close previous standalone retriever:", e)

rag = RAGPipeline(
    vectorstore_path=QDRANT_LOCAL_PATH,
    collection_name=settings.COLLECTION_NAME,
    cache_file=CACHE_FILE,
    use_cache=True
)

result = rag.ask("سواد مالی یعنی چه؟", top_k=5)

print("From cache:", result["from_cache"])
print()
print("Answer:")
print(result["answer"])

print()
print("Sources:")
for source in result["sources"]:
    print("-", source["title"])

No previous RAG pipeline found
Previous standalone retriever closed
From cache: True

Answer:
سواد مالی، کسب دانش کافی، داشتن نگرش صحیح و اجرای درست آموزه‌های مالی است.  
هدف این دانش، دستیابی به امنیت و رسیدن به آزادی مالی است.  
سواد مالی شامل مفاهیمی مانند خرج و پس‌انداز، اعتبار و بدهی، سرمایه‌گذاری و مدیریت ریسک می‌شود.

Sources:
- فیل سواد مالی
- سیر مطالعاتی سواد مالی؛ گام نخست
- سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟


In [15]:
result = rag.ask("آیا سواد مالی فقط سرمایه‌گذاری است؟", top_k=5)

print("From cache:", result["from_cache"])
print()
print(result["answer"])

From cache: True

خیر، سواد مالی فقط سرمایه‌گذاری نیست.  
سواد مالی شامل شش سرفصل اصلی است: خرج و پس‌انداز، اعتبار و بدهی، کاریابی و درآمد، سرمایه‌گذاری، مدیریت ریسک و بیمه و تصمیم‌های مالی.  
این مفهوم یک کل درهم تنیده است و فقط به یک بخش محدود مربوط نمی‌شود.


In [16]:
result = rag.ask("چرا آموزش سواد مالی برای کودکان مهم است؟", top_k=5)

print("From cache:", result["from_cache"])
print()
print("Answer:")
print(result["answer"])

print()
print("Sources:")
for source in result["sources"]:
    print("-", source["title"])

From cache: False

Answer:
آموزش سواد مالی برای کودکان مهم است، چون به آن‌ها کمک می‌کند تا از پیش‌بینی و مدیریت امور مالی به‌طور صحیح برخوردار شوند. این آموزش به آن‌ها کمک می‌کند تا در آینده با دانش کافی، تصمیم‌های درستی در مورد پس‌انداز، سرمایه‌گذاری و مدیریت بدهی‌ها بگیرند. همچنین باعث می‌شود تا در برابر بحران‌های مالی مقاومت کنند و کیفیت زندگی خود را ارتقا دهند.

Sources:
- سیر مطالعاتی سواد مالی؛ گام نخست
- سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟
- فیل سواد مالی


In [17]:
result = rag.ask("چرا آموزش سواد مالی برای کودکان مهم است؟", top_k=5)

print("From cache:", result["from_cache"])
print()
print("Answer:")
print(result["answer"])

print()
print("Sources:")
for source in result["sources"]:
    print("-", source["title"])

From cache: True

Answer:
آموزش سواد مالی برای کودکان مهم است، چون به آن‌ها کمک می‌کند تا از پیش‌بینی و مدیریت امور مالی به‌طور صحیح برخوردار شوند. این آموزش به آن‌ها کمک می‌کند تا در آینده با دانش کافی، تصمیم‌های درستی در مورد پس‌انداز، سرمایه‌گذاری و مدیریت بدهی‌ها بگیرند. همچنین باعث می‌شود تا در برابر بحران‌های مالی مقاومت کنند و کیفیت زندگی خود را ارتقا دهند.

Sources:
- سیر مطالعاتی سواد مالی؛ گام نخست
- سواد مالی برای خانم‌ها؛ واقعاً ضروری است؟
- فیل سواد مالی


In [ ]:
try:
    retriever.client.close()
    print("Standalone retriever closed")
except NameError:
    pass

rag.close()
print("RAG pipeline closed")

NameError: name 'rag' is not defined

: 